# Deliverable 3 · Tres agentes sobre normativa UdeC · v3

Implementación nueva: investigador, redactor y auditor, todos con Qwen3-4B.
No utiliza los prototipos previos de D3. Integra el corpus, recuperador y controles de D2.

**Piloto v3 pendiente de evaluación en T4.** La versión anterior obtuvo 1/4 con agentes y 2/4 con D2
en los cuatro casos de diagnóstico. Este cuaderno incorpora las correcciones; la corrida anterior se conserva en el repositorio.
La aceptación del auditor no es una calificación. Reinicia la sesión si usaste el cuaderno anterior.
Este cuaderno genera respuestas reales: requiere GPU CUDA, descarga modelos públicos y conserva trazas.
1. Selecciona GPU en Colab y ejecuta preparación y carga.
2. «Ejecutar todo» corre por defecto el piloto de cuatro preguntas (12 respuestas finales).
   La consulta individual es opcional. Los mensajes entre agentes se muestran en pantalla.
3. La sección 8 crea el ZIP y solicita su descarga. Si el navegador no la permite, descarga
   el archivo desde el panel Archivos de Colab. No cierres la sesión antes de conservarlo.
4. El lote completo y las ablaciones se activan expresamente después de comprobar el piloto.

## 1. Dependencias (reinicia la sesión si ya había otras versiones cargadas)

In [ ]:
import sys, subprocess, importlib.metadata
PINNED = {'transformers':'4.57.6', 'accelerate':'1.11.0', 'bitsandbytes':'0.48.1',
          'sentence-transformers':'5.1.2', 'tokenizers':'0.22.2'}
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                      *[p+'=='+v for p,v in PINNED.items()]])
for module, package in [(p.replace('-', '_'), p) for p in PINNED] + [
    ('numpy','numpy'), ('scipy','scipy'), ('torch','torch'), ('sklearn','scikit-learn'),
    ('huggingface_hub','huggingface-hub')]:
    loaded = sys.modules.get(module)
    if loaded is not None and hasattr(loaded, '__version__'):
        if loaded.__version__ != importlib.metadata.version(package):
            raise RuntimeError('Reinicia la sesión: versión anterior en memoria de ' + package)
print('Dependencias instaladas. No se cargó ningún modelo todavía.')

## 2. Restaurar código, prompts externos y corpus incluidos

In [ ]:
BUNDLE_B64 = (
    '...'
)
EXPECTED = '230ba7a0f07ee4caae4e938382e94ad746e68f1fdf027f76def74819816633ad'
FILE_HASHES = {'d3_agentes/__init__.py': '6b87d25aced008a504e6715f67721e823e173c40674daf21dacb822332c59566', 'd3_agentes/bridge.py': '0e15d6153defdb4bc77c0c2a99d32e28e34a31b5f24c97b94fd0e06c23924ca1', 'd3_agentes/common.py': '76f9cb1d436e46f095d37a70ddd9b1ab8c8da0f8e1ce5ee1f8f2f1831077bb20', 'd3_agentes/core.py': 'be4258f7e3a20a721fb5135ac6b87fb6fa045148d287a4809745895370efc74e', 'd3_agentes/downloads.py': '9345bffa531e1fb55dcd6d572f8afbc9c324972e29c88103aa297486042924cc', 'd3_agentes/experiment.py': '4de698bf535d87b4a7392429fad1649f3dbea539190f15b3da451d7a75d72363', 'd3_agentes/runtime.py': '112401003b7051b2f462f0496bfa1f6a313a7fa01c7c7a262930837fbc04ab6e', 'd3_agentes/tools.py': 'f075926974535b9b03aa8eaf431b0a803af13d81444b08f136818643a09ddab0', 'prompts/auditor.txt': '69aa2186f17c3a4b86799f4435060ef86865e8be00a37e49f4249bc1df7be9b2', 'prompts/completitud.txt': '3478a7b670674bc5b507b390cb7092080e97f0287cc0f973a440ee3141978523', 'prompts/descomposicion.txt': 'b51dba5d9069532badd3cc8e723aa854e3465885cc08e95401db1fe27f8fe4ef', 'prompts/investigador.txt': '0bcc8694aa10ecdcfa5c17438071a7390cd77a2a355141191e13b5704ee1c9ea', 'prompts/redactor.txt': '9c177dd67280afa56076b640cabdccd9b45b642ab35d859e98277d1d1317812b', 'prompts/verificacion.txt': '5d41fd33e5332afb28a1b2922a6385ca3a98135d9b913d18ea6fef3f4080b7fb', 'd2_runtime/experimento.py': 'd20ce66a54c7791a480e0d9466e1740b34a8c5198df94ec52c1d4868219e22d0', 'd2_runtime/sistema.py': 'be9ff8714f6580b30618ef4d45c98e744463da620e98721701dadfe9521fe130', 'd2_runtime/recuperador_portable.py': 'a04c720b0cf25eb8010996dd28f70f8a08806b6c556e0be6bc2ea3a6fb4dd056', 'data/corpus.json': '0a31ff812a1969ff81bbc895feaba490e3bd027b092e52101296e3c872719400', 'data/fragmentos_busqueda.json': 'c125615c1c3cddb23d69c3b2b93d7a73e076c90fe861cc217b59f293aa9b14cd', 'data/preguntas.json': '89084eb950e1c1bca00c0a90adbf737a500a978cf12fe61bf70f2325ccf15d12', 'data/configuracion.json': 'd5dd7936c24d811b302cc961b926735a8626708a94fc65b21424dad03eaab310'}

import base64, gzip, hashlib, json, sys
from pathlib import Path
packed = base64.b64decode(BUNDLE_B64, validate=True)
if hashlib.sha256(packed).hexdigest() != EXPECTED:
    raise ValueError('Paquete alterado.')
sources = json.loads(gzip.decompress(packed))
if set(sources) != set(FILE_HASHES):
    raise ValueError('Inventario del paquete incorrecto.')
WORKDIR = Path.cwd() / ('d3_nuevo_runtime_' + EXPECTED[:16])
for name, text in sources.items():
    path = Path(name)
    if path.is_absolute() or '..' in path.parts:
        raise ValueError('Ruta no permitida.')
    if hashlib.sha256(text.encode()).hexdigest() != FILE_HASHES[name]:
        raise ValueError('Fuente alterada: ' + name)
    target = WORKDIR / path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(text, encoding='utf-8')
previous = sys.modules.get('d3_agentes')
if previous is not None and Path(previous.__file__).resolve().parent != (WORKDIR / 'd3_agentes').resolve():
    raise RuntimeError('Reinicia la sesión para cargar esta versión de D3.')
if str(WORKDIR.resolve()) not in sys.path:
    sys.path.insert(0, str(WORKDIR.resolve()))
del BUNDLE_B64, sources, packed
print('Código y prompts restaurados en', WORKDIR)

## 3. Cargar una sola instancia Qwen y el índice E5

In [ ]:
from d3_agentes.runtime import load, execute
from d3_agentes.common import VARIANTS
from d3_agentes.experiment import export, audit_zip, sealed_load
USAR_DRIVE = False
if USAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT = Path('/content/drive/MyDrive/GenIA_D3_Agentes/runs')
else:
    OUTPUT = Path.cwd() / 'd3_agentes_runs'
ENV = load(OUTPUT)
print('Preguntas de desarrollo:', len(ENV['questions']))

## 4. Consulta individual con mensajes visibles

Compara baseline, D2 y tres agentes. Repetir exactamente entradas/configuración reutiliza checkpoints
y lo indica. Para una repetición nueva cambia ETIQUETA y conserva todos los intentos.

In [ ]:
PREGUNTA = ''
ETIQUETA = 'consulta-v3'
if PREGUNTA.strip():
    RUN_DIR = None  # Evita conservar la ruta de otra corrida si esta ejecución se interrumpe.
    RUN_DIR = execute(ENV, [{'id':1, 'pregunta':PREGUNTA, 'categoria':'nueva'}], OUTPUT,
                      variants=VARIANTS[:3], tag=ETIQUETA, verbose=True)
else:
    print('Consulta individual omitida. La sección 5 ejecutará el piloto por defecto.')

## 5. Piloto v3 de cuatro casos conocidos

**Versión corregida tras dos pilotos.** Reinicia la sesión de Colab antes de cargarla.
Se aclaró que las fuentes se transfieren automáticamente, se acotaron los mensajes y se mejoró la reparación de formato.
Su calidad todavía debe verificarse con esta nueva ejecución en T4.

P05: caso correcto en D2; P25: omisión; P38: recuperación/cita; P41: premisa falsa.
Se eligieron para comprobar comportamientos distintos, no para estimar exactitud.
Conserva rechazos, JSON inválido y regresiones. La aceptación interna no sustituye a la pauta externa.
**Activado por defecto:** al ejecutar todas las celdas se generan 12 respuestas finales
(4 preguntas × baseline/D2/agentes). La sección 8 exporta y descarga el ZIP.

In [ ]:
EJECUTAR_PILOTO = True
if EJECUTAR_PILOTO:
    RUN_DIR = None
    RUN_DIR = execute(ENV, [q for q in ENV['questions'] if q['id'] in [5,25,38,41]], OUTPUT,
                      variants=VARIANTS[:3], tag='piloto-v3', verbose=True)
    print('Piloto terminado. ZIP guardado:', RUN_DIR.with_suffix('.zip'))
    print('Continúa hasta la sección 8 para descargarlo.')
else:
    print('Piloto desactivado.')

## 6. Lote completo con tres ablaciones

Activa solo después de comprobar el piloto y congelar esta versión. Produce 300 respuestas finales
(50 × 6 condiciones), con múltiples llamadas por sistema de agentes. Puede exceder una sesión gratuita;
Drive y checkpoints permiten continuar. Estas 50 preguntas son desarrollo, no generalización.
Ablaciones: sin descomposición explícita del investigador, sin lista de completitud del redactor,
sin verificación explícita por afirmación del auditor. Se conservan agentes, herramientas e interfaces.

In [ ]:
EJECUTAR_LOTE = False
if EJECUTAR_LOTE:
    RUN_DIR = None
    RUN_DIR = execute(ENV, ENV['questions'], OUTPUT, variants=VARIANTS,
                      tag='lote-v3', verbose=False)
else:
    print('Lote desactivado; no se han generado resultados de ablaciones.')

## 7. Inspeccionar una traza completa

In [ ]:
ID = 25
VARIANTE = 'agentes_completo'
if globals().get('RUN_DIR') is not None:
    path = RUN_DIR / 'resultados' / f'P{ID:03}_{VARIANTE}.json'
    if path.exists():
        saved = sealed_load(path)
        for event in saved['resultado']['traza']:
            print(json.dumps(event, ensure_ascii=False, indent=2))
        print('Respuesta final:', saved['resultado']['respuesta_modelo'])
    else:
        print('Ese caso aún no está completo en la corrida seleccionada.')
else:
    print('Selecciona una corrida después de generar.')

## 8. Descargar o recuperar una corrida parcial

Esta celda encuentra las corridas guardadas aunque RUN_DIR no haya quedado asignado. Por defecto
conserva todas: si hay varias, las reúne como ZIP individuales dentro de una descarga.
CARPETA_A_EXPORTAR permite elegir solo una. Un ZIP parcial conserva las llamadas ya realizadas.
Si el navegador no inicia la descarga, usa el panel Archivos de Colab y la ruta que se imprime.

In [ ]:
from d3_agentes.downloads import prepare_download
CARPETA_A_EXPORTAR = ''  # Vacío: recuperar todas las corridas de OUTPUT.
ZIP = prepare_download(OUTPUT, CARPETA_A_EXPORTAR or None)
if ZIP is not None:
    print('ZIP listo:', ZIP.resolve(), '| bytes:', ZIP.stat().st_size, flush=True)
    print('Si no se descarga automáticamente: panel Archivos > este ZIP > Descargar.', flush=True)
    try:
        from google.colab import files
        files.download(str(ZIP))
    except Exception as exc:
        print('No se pudo solicitar la descarga automática:', str(exc))
        print('El ZIP sigue guardado en:', ZIP.resolve())

## Evaluación y entrega

Aplicar la pauta D2 fuera de la generación, con motivos y hashes por respuesta. Los CSV de preguntas
conocidas son compatibles con su evaluador. Para preguntas nuevas hay que elaborar referencias.
No reutilizar etiquetas antiguas para nuevas salidas. Reportar aciertos, regresiones, límites,
errores de formato, tokens y tiempo, junto con las ablaciones; consultar EVALUACION.md del repositorio.
El informe y el video finales se preparan después de obtener resultados reales.